# Reproducible Analysis of the Sulawesi EV Charging Station Recommendation System

**Computational companion for journal preparation — analysis environment version 0.18.0**

This notebook explains and verifies four main components of the system:

1. characteristics and consolidation of the Sulawesi EV charging station dataset;
2. connector compatibility and public/dealer charging networks;
3. the energy model and Dynamic Programming (DP) with state `(node, SOC)`; and
4. results from six historical baseline scenarios, seven historical sensitivity scenarios, and nine live total-detour sensitivity scenarios.

> All analyses in this notebook run **offline**. No Google Maps API is called, so running all cells does not consume Google Cloud quota or incur billing. The live results analyzed here are historical: the baseline was produced by application version 0.9.2 and the sensitivity study by version 0.10.0, not by version 0.18.0.

## How to Read the Results and Their Interpretation Boundaries

- The dataset is a snapshot of locations, connectors, and coordinates; real-time charger operating status is unavailable.
- Hyundai, Wuling, and Toyota/Lexus labels are treated as **conditional dealer networks**, not as guaranteed access. Users must still confirm the operator's policy.
- The historical 0.9.2/0.10.0 experiments use **CCS2 only** and a 300 km vehicle. The 0.18.0 total-detour experiment separately evaluates a 430 km Combo 2 vehicle with caps of 10/20/30 km on three corridors; it must not be merged with the historical snapshots.
- `graph_disconnected` means that no graph path was formed under the experiment's parameters, connector filter, corridor candidates, and road data. It does not prove that the physical journey is impossible.
- Reported times come from Google Routes. **Charging time is not calculated**.
- Runtime includes network latency during live experiments. DP state and transition statistics should therefore be used more cautiously when discussing algorithmic load.
- Zero SOC violations means that a feasible solution satisfies the model constraints; it is not proof of real-world vehicle safety.
- The dataset's original source, acquisition date, and license remain undocumented; consult `dataset_metadata.json` before publication or redistribution.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from html import escape
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import HTML, display
except ImportError:  # Enables cell verification without Jupyter/IPython.
    class HTML:
        def __init__(self, data):
            self.data = data

    def display(value):
        if isinstance(value, HTML):
            print('[SVG visualization is ready for display in Jupyter]')
        else:
            print(value)

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 160)

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'release_manifest.json').is_file() and (candidate / 'dataset_spklu_sulawesi.csv').is_file():
            return candidate
    raise FileNotFoundError(
        'Project root was not found. Run the notebook from within the spklu-sulawesi repository.'
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
NOTEBOOK_DATA = PROJECT_ROOT / 'notebooks' / 'data'
DATASET_PATH = PROJECT_ROOT / 'dataset_spklu_sulawesi.csv'
MANIFEST_PATH = PROJECT_ROOT / 'release_manifest.json'
RANGE_REFERENCE_PATH = PROJECT_ROOT / 'research' / 'vehicle_range_reference.json'
EXPORT_FIGURES = False
FIGURE_DIR = PROJECT_ROOT / 'notebooks' / 'figures'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.services.dataset import (
    CHARGING_NETWORK_LABELS,
    CHARGING_NETWORK_ORDER,
    CONNECTOR_ORDER,
    load_station_catalog,
    node_is_eligible,
)
from app.services.energy import EnergyParameters, SocDiscretizer
from app.services.vehicle_reference import load_vehicle_range_reference

print(f'Project root: {PROJECT_ROOT}')
print('Analysis mode: OFFLINE — no Google Maps API requests')

In [ ]:
PALETTE = ['#0f766e', '#2563eb', '#d97706', '#dc2626', '#7c3aed', '#0891b2']

def show_table(frame: pd.DataFrame, decimals: int = 3):
    display(frame.round(decimals).reset_index(drop=True))

def show_svg(svg: str, filename: str | None = None):
    if EXPORT_FIGURES and filename:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        (FIGURE_DIR / filename).write_text(svg, encoding='utf-8')
    display(HTML(svg))

def horizontal_bar_svg(labels, values, title, value_label='', color='#0f766e'):
    labels = [str(label) for label in labels]
    values = [float(value) for value in values]
    width, left, right = 900, 255, 85
    row_height, top, bottom = 38, 58, 44
    height = top + bottom + row_height * len(labels)
    plot_width = width - left - right
    maximum = max(values) if values and max(values) > 0 else 1.0
    pieces = [
        f'<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{escape(title)}">',
        '<style>text{font-family:Arial,sans-serif;fill:#173042}.title{font-size:20px;font-weight:700}.label{font-size:14px}.value{font-size:13px;font-weight:700}</style>',
        f'<text class="title" x="20" y="30">{escape(title)}</text>',
    ]
    for index, (label, value) in enumerate(zip(labels, values)):
        y = top + index * row_height
        bar_width = value / maximum * plot_width
        shown = f'{value:,.2f}'.rstrip('0').rstrip('.')
        pieces.extend([
            f'<text class="label" x="{left - 12}" y="{y + 17}" text-anchor="end">{escape(label)}</text>',
            f'<rect x="{left}" y="{y}" width="{bar_width:.2f}" height="24" rx="4" fill="{color}" opacity="0.88"/>',
            f'<text class="value" x="{min(left + bar_width + 8, width - 70):.2f}" y="{y + 17}">{shown}{escape(value_label)}</text>',
        ])
    pieces.append('</svg>')
    return ''.join(pieces)

def station_scatter_svg(frame: pd.DataFrame, title: str):
    width, height = 900, 590
    left, right, top, bottom = 75, 210, 55, 65
    plot_width, plot_height = width - left - right, height - top - bottom
    x_min, x_max = frame['longitude'].min(), frame['longitude'].max()
    y_min, y_max = frame['latitude'].min(), frame['latitude'].max()
    x_pad, y_pad = (x_max - x_min) * 0.04, (y_max - y_min) * 0.04
    x_min, x_max = x_min - x_pad, x_max + x_pad
    y_min, y_max = y_min - y_pad, y_max + y_pad

    def sx(value):
        return left + (value - x_min) / (x_max - x_min) * plot_width

    def sy(value):
        return top + (y_max - value) / (y_max - y_min) * plot_height

    provinces = sorted(frame['province'].unique())
    colors = {province: PALETTE[index % len(PALETTE)] for index, province in enumerate(provinces)}
    pieces = [
        f'<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{escape(title)}">',
        '<style>text{font-family:Arial,sans-serif;fill:#173042}.title{font-size:20px;font-weight:700}.axis{font-size:12px}.legend{font-size:13px}</style>',
        f'<text class="title" x="20" y="30">{escape(title)}</text>',
        f'<rect x="{left}" y="{top}" width="{plot_width}" height="{plot_height}" fill="#f8fafc" stroke="#cbd5e1"/>',
    ]
    for _, row in frame.iterrows():
        pieces.append(
            f'<circle cx="{sx(row.longitude):.2f}" cy="{sy(row.latitude):.2f}" r="3.6" fill="{colors[row.province]}" opacity="0.76"/>'
        )
    pieces.extend([
        f'<text class="axis" x="{left + plot_width / 2}" y="{height - 20}" text-anchor="middle">Longitude</text>',
        f'<text class="axis" transform="translate(20 {top + plot_height / 2}) rotate(-90)" text-anchor="middle">Latitude</text>',
    ])
    legend_x = left + plot_width + 25
    for index, province in enumerate(provinces):
        y = top + 20 + index * 34
        pieces.extend([
            f'<circle cx="{legend_x}" cy="{y}" r="6" fill="{colors[province]}"/>',
            f'<text class="legend" x="{legend_x + 13}" y="{y + 5}">{escape(province)}</text>',
        ])
    pieces.append('</svg>')
    return ''.join(pieces)

def soc_profile_svg(legs: pd.DataFrame, minimum_soc: float = 20.0):
    width, height = 900, 500
    left, right, top, bottom = 75, 210, 55, 65
    plot_width, plot_height = width - left - right, height - top - bottom
    profiles = {}
    for region, group in legs.sort_values(['region', 'sequence']).groupby('region'):
        distance = 0.0
        points = [(0.0, float(group.iloc[0]['departure_soc_percent']))]
        for _, leg in group.iterrows():
            distance += float(leg['road_distance_km'])
            points.append((distance, float(leg['arrival_soc_percent'])))
            if leg['sequence'] != group['sequence'].max():
                next_departure = float(group.loc[group['sequence'] == leg['sequence'] + 1, 'departure_soc_percent'].iloc[0])
                points.append((distance, next_departure))
        profiles[region] = points
    max_distance = max(x for points in profiles.values() for x, _ in points)

    def sx(value):
        return left + value / max_distance * plot_width

    def sy(value):
        return top + (100 - value) / 100 * plot_height

    pieces = [
        f'<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="SOC profiles for feasible baseline scenarios">',
        '<style>text{font-family:Arial,sans-serif;fill:#173042}.title{font-size:20px;font-weight:700}.axis{font-size:12px}.legend{font-size:13px}</style>',
        '<text class="title" x="20" y="30">SOC Profiles for Feasible Baseline Scenarios</text>',
        f'<rect x="{left}" y="{top}" width="{plot_width}" height="{plot_height}" fill="#f8fafc" stroke="#cbd5e1"/>',
        f'<line x1="{left}" y1="{sy(minimum_soc):.2f}" x2="{left + plot_width}" y2="{sy(minimum_soc):.2f}" stroke="#dc2626" stroke-dasharray="7 5"/>',
        f'<text class="axis" x="{left + 8}" y="{sy(minimum_soc) - 7:.2f}" fill="#dc2626">Minimum SOC {minimum_soc:.0f}%</text>',
    ]
    for index, (region, points) in enumerate(profiles.items()):
        color = PALETTE[index % len(PALETTE)]
        coordinates = ' '.join(f'{sx(x):.2f},{sy(y):.2f}' for x, y in points)
        pieces.append(f'<polyline points="{coordinates}" fill="none" stroke="{color}" stroke-width="3"/>')
        for x, y in points:
            pieces.append(f'<circle cx="{sx(x):.2f}" cy="{sy(y):.2f}" r="4" fill="{color}"/>')
        legend_y = top + 22 + index * 32
        pieces.extend([
            f'<line x1="{left + plot_width + 20}" y1="{legend_y}" x2="{left + plot_width + 47}" y2="{legend_y}" stroke="{color}" stroke-width="3"/>',
            f'<text class="legend" x="{left + plot_width + 55}" y="{legend_y + 5}">{escape(region)}</text>',
        ])
    pieces.extend([
        f'<text class="axis" x="{left + plot_width / 2}" y="{height - 20}" text-anchor="middle">Cumulative road distance (km)</text>',
        f'<text class="axis" transform="translate(20 {top + plot_height / 2}) rotate(-90)" text-anchor="middle">SOC (%)</text>',
        '</svg>',
    ])
    return ''.join(pieces)

print('Table and SVG visualization helpers are ready.')

## 1. Provenance and Artifact Integrity

The following cell checks the dataset hash against the release manifest and every research snapshot hash against the provenance metadata. If a file changes without an intentional metadata update, execution stops to prevent journal figures from mixing different versions. Original JSON reports are also checked when they remain available under `reports/generated/`. The tracked CSV snapshots are sufficient to reproduce the offline tables and figures, but a fresh clone cannot independently repeat the raw-report transformation without the source reports.

In [ ]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
provenance = json.loads((NOTEBOOK_DATA / 'provenance.json').read_text(encoding='utf-8'))
catalog = load_station_catalog(DATASET_PATH)
catalog_summary = catalog.summary()
range_reference, range_reference_summary = load_vehicle_range_reference(RANGE_REFERENCE_PATH)

baseline = pd.read_csv(NOTEBOOK_DATA / 'baseline_results.csv')
sensitivity = pd.read_csv(NOTEBOOK_DATA / 'sensitivity_results.csv')
detour_sensitivity = pd.read_csv(NOTEBOOK_DATA / 'detour_sensitivity_results.csv')
baseline_legs = pd.read_csv(NOTEBOOK_DATA / 'baseline_itinerary_legs.csv')

REGION_LABELS_EN = {
    'Sulawesi Selatan': 'South Sulawesi',
    'Sulawesi Tengah': 'Central Sulawesi',
    'Sulawesi Tenggara': 'Southeast Sulawesi',
    'Sulawesi Utara': 'North Sulawesi',
    'Sulawesi Barat': 'West Sulawesi',
    'Gorontalo': 'Gorontalo',
}
SCENARIO_LABELS_EN = {
    'Makassar ke Rantepao': 'Makassar to Rantepao',
    'Palu ke Moutong': 'Palu to Moutong',
    'Kolaka ke Kendari': 'Kolaka to Kendari',
    'Bintauna ke Manado': 'Bintauna to Manado',
    'Polewali ke Mamuju': 'Polewali to Mamuju',
    'Marisa ke Kota Gorontalo': 'Marisa to Gorontalo City',
    'Baseline alpha 0,9; koridor 10 km; SOC 5%': 'Baseline: alpha 0.9; 10 km corridor; 5% SOC interval',
    'Alpha 0,8': 'Alpha 0.8',
    'Alpha 1,0': 'Alpha 1.0',
    'Koridor 5 km': '5 km corridor',
    'Koridor 15 km': '15 km corridor',
    'Diskretisasi SOC 2,5%': '2.5% SOC discretization',
    'Diskretisasi SOC 10%': '10% SOC discretization',
}
for frame in (baseline, sensitivity):
    frame['region_display'] = frame['region'].map(REGION_LABELS_EN).fillna(frame['region'])
    frame['scenario_name_display'] = frame['scenario_name'].map(SCENARIO_LABELS_EN).fillna(frame['scenario_name'])
baseline_legs_display = baseline_legs.copy()
baseline_legs_display['region'] = baseline_legs_display['region'].map(REGION_LABELS_EN).fillna(baseline_legs_display['region'])

assert catalog.source_sha256 == manifest['dataset']['sha256']
assert catalog.source_sha256 == provenance['dataset']['sha256']
assert catalog.source_row_count == manifest['dataset']['source_rows']
assert catalog.logical_node_count == manifest['dataset']['logical_nodes']
assert manifest['application_version'] == provenance['analysis']['application_version']
assert sha256_file(RANGE_REFERENCE_PATH) == manifest['algorithm']['range_reference']['sha256']
assert range_reference_summary['selected_baseline_maximum_range_km'] == manifest['algorithm']['reference_maximum_range_km']

integrity_rows = [
    {
        'Artifact': DATASET_PATH.name,
        'Status': 'matches manifest',
        'SHA-256': catalog.source_sha256,
    }
]
integrity_rows.append({
    'Artifact': str(RANGE_REFERENCE_PATH.relative_to(PROJECT_ROOT)),
    'Status': 'scientific baseline recomputed',
    'SHA-256': sha256_file(RANGE_REFERENCE_PATH),
})
for filename, metadata in provenance['tracked_snapshots'].items():
    path = NOTEBOOK_DATA / filename
    actual_hash = sha256_file(path)
    assert actual_hash == metadata['sha256'], filename
    frame_rows = len(pd.read_csv(path))
    assert frame_rows == metadata['rows'], filename
    integrity_rows.append({'Artifact': filename, 'Status': 'matches provenance', 'SHA-256': actual_hash})

for report_name, metadata in provenance['source_reports'].items():
    source_path = PROJECT_ROOT / metadata['path']
    if source_path.is_file():
        actual_hash = sha256_file(source_path)
        assert actual_hash == metadata['sha256'], report_name
        source_report = json.loads(source_path.read_text(encoding='utf-8'))
        assert source_report['execution']['app_version'] == metadata['source_application_version'], report_name
        integrity_rows.append({'Artifact': metadata['path'], 'Status': 'original report verified', 'SHA-256': actual_hash})

show_table(pd.DataFrame(integrity_rows), decimals=0)
print(f"Analysis environment version: {manifest['application_version']}")
print('Result source versions: baseline 0.9.2 | sensitivity 0.10.0 | detour 0.18.0')
print(f"Baseline: {len(baseline)} scenarios | Sensitivity: {len(sensitivity)} scenarios | Detour: {len(detour_sensitivity)} scenarios")

## 2. Dataset Characteristics and Spatial Coverage

Each CSV row is retained as one charger unit. Units sharing coordinates to six decimal places are consolidated into one location node for the routing algorithm. Unit-level information is therefore preserved while the graph avoids duplicate nodes at identical coordinates.

In [ ]:
dataset_overview = pd.DataFrame([
    {'Indicator': 'Source rows/units', 'Value': catalog_summary['source_rows']},
    {'Indicator': 'Logical location nodes', 'Value': catalog_summary['logical_nodes']},
    {'Indicator': 'Multi-unit nodes', 'Value': catalog_summary['multi_unit_node_count']},
    {'Indicator': 'Provinces covered', 'Value': len(catalog_summary['province_counts'])},
])
show_table(dataset_overview, decimals=0)

province_counts = pd.DataFrame(
    catalog_summary['province_counts'].items(), columns=['Province', 'Node count']
).sort_values('Node count', ascending=False)
province_counts['Province'] = province_counts['Province'].map(REGION_LABELS_EN).fillna(province_counts['Province'])
show_table(province_counts, decimals=0)
show_svg(
    horizontal_bar_svg(province_counts['Province'], province_counts['Node count'], 'EV charging station nodes by province'),
    '01_province_distribution.svg',
)

In [ ]:
nodes_df = pd.DataFrame([node.to_dict(include_units=False) for node in catalog.nodes])
nodes_df['province'] = nodes_df['province'].map(REGION_LABELS_EN).fillna(nodes_df['province'])
nodes_df['connectors_text'] = nodes_df['connectors'].map(', '.join)
nodes_df['networks_text'] = nodes_df['charging_networks'].map(', '.join)
show_svg(
    station_scatter_svg(nodes_df, 'Spatial distribution of 146 Sulawesi charging-station nodes'),
    '02_station_node_distribution.svg',
)
print('Note: this visualization is a coordinate scatter plot, not a road map or a substitute for Google route validation.')

In [ ]:
connector_counts = pd.DataFrame({
    'Connector': list(CONNECTOR_ORDER),
    'Location nodes': [catalog_summary['connector_node_counts'][name] for name in CONNECTOR_ORDER],
    'Source units': [catalog_summary['connector_unit_counts'][name] for name in CONNECTOR_ORDER],
})
show_table(connector_counts, decimals=0)
show_svg(
    horizontal_bar_svg(connector_counts['Connector'], connector_counts['Location nodes'], 'Location nodes by connector type'),
    '03_connector_distribution.svg',
)

network_connector = pd.DataFrame.from_dict(
    catalog_summary['network_connector_node_counts'], orient='index'
)[list(CONNECTOR_ORDER)]
network_labels_en = {'PUBLIC': 'Public charging stations', 'HYUNDAI': 'Hyundai', 'WULING': 'Wuling', 'TOYOTA': 'Toyota/Lexus'}
network_connector.index = [network_labels_en[index] for index in network_connector.index]
network_connector.index.name = 'Network'
display(network_connector)

### Unit Consolidation and Dealer-Network Separation

Consolidation is based on coordinates, while compatibility remains evaluated at unit level. The system applies the following eligibility rule:

```text
eligible unit = connector matches
                AND
                (PUBLIC network or a user-selected dealer network)
```

The public network is always included. Hyundai, Wuling, and Toyota/Lexus checkboxes only extend the candidate set to the corresponding dealer networks. Because operators may change their policies, these selections do not guarantee access.

In [ ]:
multi_unit_rows = []
for node in catalog.multi_unit_nodes:
    for unit in node.units:
        multi_unit_rows.append({
            'Logical node': node.name,
            'Node ID': node.node_id,
            'Source row': unit.source_row,
            'Unit name': unit.name,
            'Connectors': ', '.join(unit.connectors),
        })
show_table(pd.DataFrame(multi_unit_rows), decimals=0)

selection_scenarios = [
    ('Combo 2: CCS2 + AC Type 2, public only', ('AC TYPE 2', 'CCS2'), ()),
    ('CCS2, public only', ('CCS2',), ()),
    ('CCS2 + Wuling checkbox', ('CCS2',), ('WULING',)),
    ('CCS2 and GB/T, public only', ('CCS2', 'GB/T'), ()),
    ('CCS2 and GB/T + Wuling checkbox', ('CCS2', 'GB/T'), ('WULING',)),
    ('AC Type 2, public only', ('AC TYPE 2',), ()),
    ('AC Type 2 + Hyundai + Toyota/Lexus', ('AC TYPE 2',), ('HYUNDAI', 'TOYOTA')),
]
eligibility_rows = []
for description, connectors, networks in selection_scenarios:
    eligible_nodes = [
        node for node in catalog.nodes
        if node_is_eligible(node, connectors, networks)
    ]
    eligibility_rows.append({
        'User selection': description,
        'Eligible node count': len(eligible_nodes),
    })
eligibility_df = pd.DataFrame(eligibility_rows)
show_table(eligibility_df, decimals=0)

assert eligibility_df.loc[eligibility_df['User selection'] == 'CCS2, public only', 'Eligible node count'].iloc[0] == 42
assert eligibility_df.loc[eligibility_df['User selection'] == 'Combo 2: CCS2 + AC Type 2, public only', 'Eligible node count'].iloc[0] == 114
assert eligibility_df.loc[eligibility_df['User selection'] == 'CCS2 + Wuling checkbox', 'Eligible node count'].iloc[0] == 42
print('Interpretation: a Combo 2 vehicle can access all 114 public nodes recorded with CCS2 or AC Type 2. The candidate optimizer prioritizes CCS2 and marks an AC-only charging stop as a fallback. Selecting Wuling together with CCS2 still does not add Wuling dealer nodes because all 17 Wuling nodes use GB/T.')

## 3. Energy Model and SOC Discretization

The model does not use battery capacity or charger power. Energy is represented as a State of Charge (SOC) percentage:

- `R_effective = R_max × alpha`
- `R_usable = ((SOC − SOC_min) / 100) × R_maks × alpha`
- `energy_distance = max(0, total_distance − ferry_distance)`
- `SOC_consumption = (energy_distance / (R_max × alpha)) × 100`

The safety factor `alpha` reduces nominal range as a conservative margin. DP rounds continuous SOC **downward** to a grid anchored at the minimum SOC, so discretization never creates artificial energy. For `FERRY`/`FERRY_TRAIN` steps, sailing distance remains reported but is excluded from traction consumption; schedules and vehicle access remain conditional.

In [ ]:
range_reference_df = pd.DataFrame([
    {
        'Manufacturer': vehicle['manufacturer'],
        'Model family': vehicle['model_family'],
        'Representative WLTP range (km)': vehicle['representative_range_km'],
        'Rule': vehicle['representative_rule'],
    }
    for vehicle in range_reference['vehicles']
])
show_table(range_reference_df)
print('Sample median:', range_reference_summary['median_range_km'], 'km')
print('Rounded candidate baseline:', range_reference_summary['selected_baseline_maximum_range_km'], 'km')

reference_parameters = EnergyParameters(
    maximum_range_km=430,
    minimum_soc_percent=20,
    target_soc_percent=80,
    safety_factor=0.9,
    soc_step_percent=5,
)
current_soc = 80
discretizer = SocDiscretizer(reference_parameters)
ferry_trip_total_km = 100
ferry_distance_km = 60
ferry_energy_distance_km = ferry_trip_total_km - ferry_distance_km
energy_example = pd.DataFrame([
    {'Quantity': 'Nominal reference range', 'Value': reference_parameters.maximum_range_km, 'Unit': 'km'},
    {'Quantity': 'Effective full range', 'Value': reference_parameters.effective_full_range_km, 'Unit': 'km'},
    {'Quantity': 'Usable range at initial SOC 80%', 'Value': reference_parameters.usable_range_km(current_soc), 'Unit': 'km'},
    {'Quantity': 'Consumption for a 100 km leg', 'Value': reference_parameters.consumption_percent(100), 'Unit': '% SOC'},
    {'Quantity': 'Arrival SOC after a 100 km leg from 80%', 'Value': reference_parameters.arrival_soc_percent(80, 100), 'Unit': '% SOC'},
    {'Quantity': 'Energy distance for a 100 km trip with 60 km by ferry', 'Value': ferry_energy_distance_km, 'Unit': 'km'},
    {'Quantity': 'SOC consumption for the ferry-inclusive trip', 'Value': reference_parameters.consumption_percent(ferry_energy_distance_km), 'Unit': '% SOC'},
])
show_table(energy_example, decimals=3)
print('SOC grid:', discretizer.levels)
print('Conservative rounding example: SOC 79% ->', discretizer.quantize_down(79), '%')
assert reference_parameters.consumption_percent(ferry_energy_distance_km) < reference_parameters.consumption_percent(ferry_trip_total_km)

## 4. Recommendation Pipeline and Dynamic Programming

```text
User input
      │
      ▼
Primary Google route + ferry-step detection
      │
      ▼
BallTree radius search + corridor/connector/network filters
      │
      ▼
Forward-directed graph (origin → station → destination)
      │  Haversine pruning, ferry adjustment, then Matrix validation
      ▼
DP state (node, discrete SOC)
      │  lexicographic objective
      ▼
Continuous-SOC resimulation + recommended itinerary
```

The candidate first minimizes AC Type 2 fallback stops, then Google travel time (including detected sailing duration but excluding schedule waiting time), number of charging stops, detour, added SOC, and finally travel distance. Total graph detour is also constrained to 20 km; transitions above the cap are pruned before comparison. The next cell uses production classes on a small synthetic graph to demonstrate how DP selects charging without calling any external service.

In [ ]:
from app.services.graph import (
    DESTINATION_NODE_ID, ORIGIN_NODE_ID, GraphBuildStats, GraphEdge, GraphNode, TravelGraph
)
from app.services.optimizer import optimize_itinerary

def demo_node(node_id, kind, progress):
    names = {'origin': 'Origin', 'destination': 'Destination', 'station': 'Example charging station'}
    return GraphNode(node_id, kind, names[kind], (0.0, progress / 100), progress)

def demo_edge(source, target, distance, duration):
    return GraphEdge(source, target, distance * 0.9, distance, duration, distance, 0.0, 250.0)

demo_graph = TravelGraph(
    nodes=(
        demo_node(ORIGIN_NODE_ID, 'origin', 0),
        demo_node('station-a', 'station', 100),
        demo_node(DESTINATION_NODE_ID, 'destination', 200),
    ),
    edges=(
        demo_edge(ORIGIN_NODE_ID, 'station-a', 100, 100),
        demo_edge('station-a', DESTINATION_NODE_ID, 100, 100),
    ),
    stats=GraphBuildStats(1, 1, 2, 0, 2, 0, 0, 0, 2, 0),
)
demo_parameters = EnergyParameters(250, 20, 80, safety_factor=1.0, soc_step_percent=5)
demo_result = optimize_itinerary(
    demo_graph, current_soc_percent=60, parameters=demo_parameters
)
assert demo_result.feasible and demo_result.itinerary is not None
demo_legs = pd.DataFrame([leg.to_dict() for leg in demo_result.itinerary.legs])
show_table(demo_legs[[
    'sequence', 'source_name', 'target_name', 'road_distance_km',
    'departure_soc_percent', 'arrival_soc_percent', 'consumption_soc_percent'
]])
show_table(pd.DataFrame([stop.to_dict() for stop in demo_result.itinerary.charging_stops]).drop(columns='station'))
print('DP statistics:', demo_result.stats.to_dict())
print('DP charges from 20% to 60% SOC at the example station so the vehicle reaches the destination exactly at the 20% minimum SOC.')

## 5. Six-Region Baseline Experiment Results

All baseline scenarios use the same parameters: CCS2, 300 km maximum range, 80% initial SOC, 20% minimum SOC, 80% target SOC, `alpha=0.9`, a 5% SOC interval, and a 10 km corridor radius. Comparing regions under fixed parameters highlights differences in CCS2 candidate density and connectivity.

In [ ]:
baseline_feasible = baseline['route_feasible'].fillna(False).astype(bool)
baseline_aggregate = pd.DataFrame([
    {'Metric': 'Completed scenarios', 'Value': int((baseline['status'] == 'completed').sum())},
    {'Metric': 'Feasible routes', 'Value': int(baseline_feasible.sum())},
    {'Metric': 'Infeasible routes', 'Value': int((~baseline_feasible).sum())},
    {'Metric': 'Feasibility rate', 'Value': baseline_feasible.mean() * 100},
    {'Metric': 'SOC violations', 'Value': int(baseline['soc_violation_count'].sum())},
    {'Metric': 'Mean stops (feasible)', 'Value': baseline.loc[baseline_feasible, 'charging_stop_count'].mean()},
    {'Metric': 'Total external requests', 'Value': int(baseline['total_external_requests'].sum())},
    {'Metric': 'Total Route Matrix elements', 'Value': int(baseline['compute_route_matrix_elements'].sum())},
])
show_table(baseline_aggregate)

baseline_view = baseline[[
    'region_display', 'route_feasible', 'reason', 'charging_stop_count', 'charging_stop_names',
    'base_route_distance_km', 'recommended_route_distance_km',
    'minimum_observed_soc_percent', 'corridor_candidate_count', 'graph_edge_count'
]].rename(columns={
    'region_display': 'Region', 'route_feasible': 'Feasible', 'reason': 'Reason',
    'charging_stop_count': 'Stop count', 'charging_stop_names': 'Selected stations',
    'base_route_distance_km': 'Base-route distance (km)',
    'recommended_route_distance_km': 'Recommended-route distance (km)',
    'minimum_observed_soc_percent': 'Minimum observed SOC (%)',
    'corridor_candidate_count': 'Corridor candidates', 'graph_edge_count': 'Graph edges',
})
show_table(baseline_view)

In [ ]:
show_svg(
    horizontal_bar_svg(baseline['region_display'], baseline['corridor_candidate_count'], 'Corridor candidates in each baseline scenario'),
    '04_baseline_candidates.svg',
)
show_svg(
    horizontal_bar_svg(baseline['region_display'], baseline['graph_edge_count'], 'Graph-edge count in each baseline scenario', color='#2563eb'),
    '05_baseline_edges.svg',
)
show_svg(
    horizontal_bar_svg(baseline['region_display'], baseline['compute_route_matrix_elements'], 'Route Matrix elements per baseline scenario', color='#d97706'),
    '06_baseline_matrix_elements.svg',
)

feasible_distance = baseline.loc[baseline_feasible, [
    'region_display', 'base_route_distance_km', 'recommended_route_distance_km', 'total_detour_km'
]].copy()
feasible_distance['distance_change_km'] = (
    feasible_distance['recommended_route_distance_km'] - feasible_distance['base_route_distance_km']
)
feasible_distance = feasible_distance.rename(columns={
    'region_display': 'Region',
    'base_route_distance_km': 'Base-route distance (km)',
    'recommended_route_distance_km': 'Recommended-route distance (km)',
    'total_detour_km': 'Total detour (km)',
    'distance_change_km': 'Distance change (km)',
})
show_table(feasible_distance)
print('Note: the recommended-route distance change relative to the base route is not identical to total_detour_km because detour is recorded at graph-edge level.')

In [ ]:
show_table(baseline_legs_display.rename(columns={
    'region': 'Region', 'sequence': 'Leg', 'source_name': 'From', 'target_name': 'To',
    'road_distance_km': 'Distance (km)', 'departure_soc_percent': 'Departure SOC (%)',
    'arrival_soc_percent': 'Arrival SOC (%)', 'consumption_soc_percent': 'SOC consumption (%)',
})[[
    'Region', 'Leg', 'From', 'To', 'Distance (km)',
    'Departure SOC (%)', 'Arrival SOC (%)', 'SOC consumption (%)'
]])
show_svg(soc_profile_svg(baseline_legs_display, minimum_soc=20), '07_baseline_soc_profiles.svg')
assert baseline_legs['arrival_soc_percent'].min() >= 20
print(f"Lowest arrival SOC across all feasible legs: {baseline_legs['arrival_soc_percent'].min():.3f}%")

## 6. One-Variable-at-a-Time Sensitivity Analysis

Seven scenarios on the Makassar–Rantepao corridor vary one parameter from the baseline: `alpha` (0.8, 0.9, 1.0), corridor radius (5, 10, 15 km), or SOC interval (2.5%, 5%, 10%). Because the analysis covers only one corridor and one run per configuration, the results describe system behavior in this case and must not be generalized as a universal pattern for Sulawesi.

In [ ]:
sensitivity_view = sensitivity[[
    'scenario_name_display', 'safety_factor', 'corridor_radius_km', 'soc_step_percent',
    'charging_stop_count', 'minimum_observed_soc_percent', 'corridor_candidate_count',
    'graph_edge_count', 'dp_processed_states', 'dp_evaluated_transitions',
    'compute_route_matrix_elements', 'runtime_ms'
]].rename(columns={
    'scenario_name_display': 'Scenario', 'safety_factor': 'Alpha',
    'corridor_radius_km': 'Radius (km)', 'soc_step_percent': 'SOC interval (%)',
    'charging_stop_count': 'Stops', 'minimum_observed_soc_percent': 'Minimum SOC (%)',
    'corridor_candidate_count': 'Candidates', 'graph_edge_count': 'Edges',
    'dp_processed_states': 'DP states', 'dp_evaluated_transitions': 'DP transitions',
    'compute_route_matrix_elements': 'Matrix elements', 'runtime_ms': 'Runtime (ms)',
})
show_table(sensitivity_view)

alpha_ids = ['sensitivitas-alpha-080', 'sensitivitas-baseline', 'sensitivitas-alpha-100']
corridor_ids = ['sensitivitas-koridor-5', 'sensitivitas-baseline', 'sensitivitas-koridor-15']
soc_ids = ['sensitivitas-soc-2-5', 'sensitivitas-baseline', 'sensitivitas-soc-10']
alpha_results = sensitivity.set_index('scenario_id').loc[alpha_ids].reset_index().sort_values('safety_factor')
corridor_results = sensitivity.set_index('scenario_id').loc[corridor_ids].reset_index().sort_values('corridor_radius_km')
soc_results = sensitivity.set_index('scenario_id').loc[soc_ids].reset_index().sort_values('soc_step_percent')

In [ ]:
show_svg(
    horizontal_bar_svg(
        [f"alpha={value:g}" for value in alpha_results['safety_factor']],
        alpha_results['charging_stop_count'],
        'Effect of the safety factor on charging-stop count',
    ),
    '08_alpha_sensitivity.svg',
)
show_svg(
    horizontal_bar_svg(
        [f"radius={value:g} km" for value in corridor_results['corridor_radius_km']],
        corridor_results['compute_route_matrix_elements'],
        'Effect of corridor radius on Route Matrix elements',
        color='#d97706',
    ),
    '09_radius_sensitivity.svg',
)
show_svg(
    horizontal_bar_svg(
        [f"interval={value:g}%" for value in soc_results['soc_step_percent']],
        soc_results['dp_evaluated_transitions'],
        'Effect of SOC interval on DP transitions',
        color='#7c3aed',
    ),
    '10_soc_interval_sensitivity.svg',
)

key_comparison = pd.DataFrame([
    {
        'Finding': 'Alpha 0.8 vs 0.9',
        'Main change': f"stops {int(alpha_results.iloc[0].charging_stop_count)} vs {int(alpha_results.iloc[1].charging_stop_count)}; edges {int(alpha_results.iloc[0].graph_edge_count)} vs {int(alpha_results.iloc[1].graph_edge_count)}",
    },
    {
        'Finding': 'Radius 5 km vs 10 km',
        'Main change': f"candidates {int(corridor_results.iloc[0].corridor_candidate_count)} vs {int(corridor_results.iloc[1].corridor_candidate_count)}; matrix elements {int(corridor_results.iloc[0].compute_route_matrix_elements)} vs {int(corridor_results.iloc[1].compute_route_matrix_elements)}",
    },
    {
        'Finding': 'SOC interval 2.5% vs 5% vs 10%',
        'Main change': 'DP transitions ' + ' vs '.join(str(int(value)) for value in soc_results['dp_evaluated_transitions']),
    },
])
show_table(key_comparison, decimals=0)

detour_display = detour_sensitivity[[
    'region', 'max_total_detour_km', 'route_feasible',
    'charging_stop_count', 'charging_stop_names', 'total_detour_km',
    'minimum_observed_soc_percent', 'detour_pruned_transitions',
    'dp_evaluated_transitions', 'compute_route_matrix_elements',
]].copy()
detour_display['region'] = detour_display['region'].map(REGION_LABELS_EN).fillna(detour_display['region'])
detour_display.columns = [
    'Region', 'Detour cap (km)', 'Feasible', 'Stops', 'Selected stations',
    'Final detour (km)', 'Minimum SOC (%)', 'Detour-pruned transitions',
    'DP transitions', 'Matrix elements',
]
show_table(detour_display)
assert len(detour_sensitivity) == 9
assert detour_sensitivity['route_feasible'].fillna(False).astype(bool).all()
assert int(detour_sensitivity['soc_violation_count'].sum()) == 0

## 7. Key Discussion Points for the Journal

1. **Location availability does not imply route connectivity.** Central, Southeast, and West Sulawesi contain corridor candidates, yet their CCS2 baseline graphs remain disconnected under the tested usable range and parameters.
2. **The conservative margin affects the itinerary.** On Makassar–Rantepao, `alpha=0.8` reduces feasible edges and produces three stops, whereas `alpha=0.9` and `1.0` produce one stop.
3. **Corridor radius affects search cost.** A 5 km radius reduces candidates and Matrix elements relative to 10 km, while the test-corridor itinerary remains unchanged. In this snapshot, 15 km adds no candidates beyond those found at 10 km.
4. **SOC interval creates an accuracy–complexity trade-off.** The 2.5% interval evaluates substantially more transitions than 5% or 10%, while the test-case itinerary remains unchanged.
5. **SOC constraints are preserved in feasible solutions.** Neither the baseline nor sensitivity results contain SOC violations, and solutions are reconstructed and resimulated using continuous SOC.
6. **Dealer access is separated from connector compatibility.** A dealer checkbox cannot make an incompatible connector valid; for example, CCS2 + Wuling still excludes Wuling units that use GB/T.
7. **Candidate connector/range defaults remain separate from historical results.** Version 0.18.0 uses the rounded 430 km median of a seven-model WLTP reference sample and treats AC Type 2 as fallback for Combo 2 vehicles.
8. **The tested detour cap was non-binding for the selected itineraries.** All 10/20/30 km scenarios were feasible with identical selected itineraries within each corridor. The caps changed DP pruning—most strongly on the medium corridor—but not the final route, stop count, or minimum SOC. Therefore 20 km is a transparent middle policy, not an empirically unique optimum.

In [ ]:
sensitivity_feasible = sensitivity['route_feasible'].fillna(False).astype(bool)
paper_facts = pd.DataFrame([
    {'Report-ready indicator': 'Source units', 'Value': catalog.source_row_count, 'Context': 'validated dataset'},
    {'Report-ready indicator': 'Logical location nodes', 'Value': catalog.logical_node_count, 'Context': 'after coordinate consolidation'},
    {'Report-ready indicator': 'CCS2 nodes', 'Value': catalog_summary['connector_node_counts']['CCS2'], 'Context': 'all networks in the dataset'},
    {'Report-ready indicator': 'Public Combo 2 eligible nodes', 'Value': eligibility_df.loc[eligibility_df['User selection'] == 'Combo 2: CCS2 + AC Type 2, public only', 'Eligible node count'].iloc[0], 'Context': 'union; CCS2 preferred, AC Type 2 fallback'},
    {'Report-ready indicator': 'Candidate total-detour cap', 'Value': manifest['algorithm']['detour_policy']['reference_max_km'], 'Context': 'km; 2 × baseline corridor radius'},
    {'Report-ready indicator': 'Detour-sensitivity feasibility', 'Value': detour_sensitivity['route_feasible'].fillna(False).astype(bool).mean() * 100, 'Context': '9 scenarios across 3 corridors (%)'},
    {'Report-ready indicator': 'Detour-sensitivity SOC violations', 'Value': detour_sensitivity['soc_violation_count'].sum(), 'Context': 'all resulting legs'},
    {'Report-ready indicator': 'Baseline feasibility', 'Value': baseline_feasible.mean() * 100, 'Context': '6 scenarios, one per region (%)'},
    {'Report-ready indicator': 'Baseline SOC violations', 'Value': baseline['soc_violation_count'].sum(), 'Context': 'all resulting legs'},
    {'Report-ready indicator': 'Sensitivity feasibility', 'Value': sensitivity_feasible.mean() * 100, 'Context': '7 Makassar–Rantepao scenarios (%)'},
    {'Report-ready indicator': 'Sensitivity SOC violations', 'Value': sensitivity['soc_violation_count'].sum(), 'Context': 'all resulting legs'},
    {'Report-ready indicator': 'Baseline external requests', 'Value': baseline['total_external_requests'].sum(), 'Context': 'Compute Routes + Matrix requests'},
    {'Report-ready indicator': 'Baseline Route Matrix elements', 'Value': baseline['compute_route_matrix_elements'].sum(), 'Context': 'not the request count'},
])
show_table(paper_facts)

assert catalog.source_row_count == 150 and catalog.logical_node_count == 146
assert int(baseline_feasible.sum()) == 3 and len(baseline) == 6
assert int(baseline['soc_violation_count'].sum()) == 0
assert sensitivity_feasible.all() and int(sensitivity['soc_violation_count'].sum()) == 0
assert int(baseline['total_external_requests'].sum()) == 40
assert int(baseline['compute_route_matrix_elements'].sum()) == 150
print('All key facts are consistent with the snapshots and pass their assertions.')

## 8. Threats to Validity and Generalization Boundaries

- **Data validity:** the dataset does not contain real-time availability, tariffs, charger power, queues, operating hours, or guaranteed dealer access.
- **Data provenance:** the dataset's original source, acquisition date, method, license, and redistribution rights remain unconfirmed. Unrecorded facts remain `null`/`unknown` rather than being inferred.
- **Energy-model validity:** consumption is assumed proportional to distance and adjusted by one safety factor; topography, weather, speed, payload, and battery degradation are not modeled explicitly.
- **Experimental validity:** the historical baseline covers only six corridors, historical alpha/radius/SOC sensitivity covers one corridor, and detour sensitivity covers three corridors at one collection time. The detour experiment does not establish 20 km as a universal optimum because every tested cap produced the same selected itinerary. Replication across other corridors and times is required for stronger generalization.
- **Dataset-version separation:** the current candidate dataset contains 150 rows and 146 logical nodes after coordinate/link revisions made after the historical live runs. Because schema-2 reports did not record a dataset hash, this current dataset must not be presented as the input to the 0.9.2/0.10.0 runs.
- **External-service dependence:** road distances and durations come from Google Routes responses at experiment time and may change with road-network or service updates.
- **Ferry crossings:** the current candidate detects ferry maneuvers and excludes sailing distance from SOC consumption, but the historical 0.9.2/0.10.0 snapshots did not evaluate this feature. Compute Route Matrix provides no ferry steps, so the graph uses overlap with the base route and the final route is revalidated. Operations, schedules, capacity, and vehicle acceptance are not guaranteed.
- **Runtime:** one measurement per scenario is insufficient for statistical performance inference because network latency is included.
- **Objective scope:** charging time is excluded in accordance with the revised proposal; the time objective represents Google travel time only.

The paper should report these limitations explicitly and describe the system as a **planning-support prototype**, not as a guarantee of journey completion or charger availability.

In [ ]:
final_validation = {
    'mode': 'offline',
    'google_api_requests_from_notebook': 0,
    'analysis_application_version': manifest['application_version'],
    'baseline_source_application_version': provenance['source_reports']['baseline']['source_application_version'],
    'sensitivity_source_application_version': provenance['source_reports']['sensitivity']['source_application_version'],
    'detour_source_application_version': provenance['source_reports']['detour_sensitivity']['source_application_version'],
    'dataset_sha256_valid': catalog.source_sha256 == manifest['dataset']['sha256'],
    'baseline_rows': len(baseline),
    'sensitivity_rows': len(sensitivity),
    'detour_sensitivity_rows': len(detour_sensitivity),
    'baseline_error_count': int((baseline['status'] != 'completed').sum()),
    'sensitivity_error_count': int((sensitivity['status'] != 'completed').sum()),
    'detour_sensitivity_error_count': int((detour_sensitivity['status'] != 'completed').sum()),
    'charging_time_included': manifest['algorithm']['charging_time_included'],
    'ferry_distance_consumes_soc': manifest['algorithm']['ferry_distance_consumes_soc'],
    'candidate_max_total_detour_km': manifest['algorithm']['detour_policy']['reference_max_km'],
}
assert final_validation['google_api_requests_from_notebook'] == 0
assert final_validation['dataset_sha256_valid']
assert final_validation['baseline_error_count'] == 0
assert final_validation['sensitivity_error_count'] == 0
assert final_validation['detour_sensitivity_error_count'] == 0
assert final_validation['charging_time_included'] is False
assert final_validation['ferry_distance_consumes_soc'] is False
assert final_validation['candidate_max_total_detour_km'] == 20
assert final_validation['baseline_source_application_version'] == '0.9.2'
assert final_validation['sensitivity_source_application_version'] == '0.10.0'
assert final_validation['detour_source_application_version'] == '0.18.0'
display(pd.DataFrame(final_validation.items(), columns=['Final check', 'Value']))
print('Notebook completed; all consistency checks passed.')

## 9. Reproduction and Use in the Paper

1. Run **Restart Kernel and Run All Cells**.
2. Confirm that every assertion passes and the final cell reports zero errors.
3. Set `EXPORT_FIGURES = True` in the setup cell to save SVG visualizations under `notebooks/figures/`.
4. Use `notebooks/data/provenance.json` to report the analysis version, source version of each result, report timestamps, and artifact hashes.
5. Separate empirical results from interpretation. Historical baseline figures come from six scenarios, historical alpha/radius/SOC sensitivity comes from one corridor, and detour sensitivity comes from nine scenarios across three corridors.
6. Do not attribute historical figures to version 0.18.0. Such a claim requires a new live experiment with separate authorization.
7. Do not rerun live experiments merely to open this notebook. If new data collection is required, follow the quota safeguards and authorization procedure in `docs/evaluation.md`.

This notebook and its snapshots are designed as a *computational companion* for the paper's Methods, Results, Discussion, and Limitations sections.